In [ ]:
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from openTSNE import TSNE
from ne_aligned_utils.utils import get_path


In [ ]:
dataset = "yao_10x"
root_path = get_path("data")
res_path = os.path.join(root_path, dataset)
fig_path = os.path.join(root_path, "figures")
os.makedirs(fig_path, exist_ok=True)

In [ ]:
#to load the pkl file:
with open(os.path.join(res_path, "yao_10x_v2_pca50.pkl"), 'rb') as f:
     pkl_file= pickle.load(f)

pkl_file.keys() #output: dict_keys(['clusters', 'clusterColors', 'clusterNames', 'pca50'])


In [ ]:
meta = pd.read_csv(os.path.join(res_path, "10x_metadata.csv"))

In [ ]:
print(meta.keys())

In [ ]:
meta["donor_sex_label"].values[:20]

In [ ]:
pkl_file['pca50'].shape

In [ ]:
pkl_file['clusters'][:10]

In [ ]:
pkl_file['clusterNames'][pkl_file['clusters'][:10]]

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))

#ax.scatter(*pkl_file['pca50'][:, :2].T, s=1, c=pkl_file['clusterColors'][pkl_file['clusters']])
ax.scatter(*pkl_file['pca50'][:, :2].T, s=1, c=meta["donor_sex_color"])

In [ ]:
def load_dict(filename_):
    with open(filename_, 'rb') as f:
        ret_di = pickle.load(f)
    return ret_di

In [ ]:
k = 15
n_epochs = 500
n_early_epochs = 250
perplexity = None
rescale_tsne = True
seed = 0

file_name = os.path.join(res_path,
                         f"tsne_bin_k_{k}_n_epochs_{n_epochs}_n_early_epochs_{n_early_epochs}_seed_{seed}_init_pca_rescale_{rescale_tsne}.pkl")

if os.path.exists(file_name):
    tsne_data = load_dict(file_name)
    embd = np.asarray(tsne_data["embds"][-1])
    embd_source = "cached t-SNE"
    print(f"Loaded {embd_source} from {file_name}")
else:
    embd = np.asarray(pkl_file["pca50"][:, :2], dtype=np.float32).copy()
    tsne_data = {
        "embds": [embd],
        "source": "pca50_2d_fallback",
        "missing_cache_path": file_name,
    }
    embd_source = "PCA50[:, :2] fallback"
    print(
        f"Cached t-SNE not found at {file_name}. "
        f"Using row-aligned {embd_source} instead."
    )

In [ ]:
n_plot = 10000  # set to None to plot all points

rng = np.random.default_rng(0)
if n_plot is not None and n_plot < len(embd):
    plot_idx = rng.choice(len(embd), size=n_plot, replace=False)
else:
    plot_idx = np.arange(len(embd))

embd_plot = embd[plot_idx]
meta_plot = meta.iloc[plot_idx].reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(15, 8))

panels = [
    ("donor_sex_color",      "donor sex",      None),
    ("cluster_color",        "cell type",      None),
    ("class_color",          "class",          "class_label"),
    ("subclass_color",       "subclass",       "subclass_label"),
    ("neighborhood_color",   "neighborhood",   "neighborhood_label"),
    ("region_color",         "region",         None),
    ("cortical_layer_color", "cortical layer", None),
]

embd_df = pd.DataFrame({"x": embd_plot[:, 0], "y": embd_plot[:, 1]})

for ax, (col, title, label_col) in zip(axes.flatten(), panels):
    ax.scatter(*embd_plot.T, s=0.1, c=meta_plot[col], alpha=1)
    ax.set_title(title)
    ax.axis("off")
    ax.set_aspect("equal")

    if label_col is not None:
        means = embd_df.assign(label=meta_plot[label_col].values).groupby("label")[["x", "y"]].mean()
        for lbl, row in means.iterrows():
            ax.text(row["x"], row["y"], lbl, fontsize=5, ha="center", va="center", fontweight="bold")

fig.tight_layout()

In [ ]:
# Look for subclasses that are separated by sex, but have stable sex ratio per cluster

In [ ]:
from sklearn.neighbors import NearestNeighbors


def sex_knn_score(pca, sex_labels, k=15, n_sample=3000, seed=42):
    """
    Normalised kNN same-sex fraction.
    0 = random mixing, 1 = perfect M/F separation (strong batch effect).
    """
    rng = np.random.default_rng(seed)
    if len(pca) > n_sample:
        idx = rng.choice(len(pca), n_sample, replace=False)
        pca, sex_labels = pca[idx], sex_labels[idx]
    nn = NearestNeighbors(n_neighbors=k + 1).fit(pca)
    _, nbrs = nn.kneighbors(pca)
    obs = (sex_labels[nbrs[:, 1:]] == sex_labels[:, None]).mean()
    f = (sex_labels == "F").mean()
    expected = f ** 2 + (1 - f) ** 2  # random baseline given class imbalance
    return float((obs - expected) / (1 - expected))


def sex_ratio_std(meta_sub):
    """Std of per-cluster female fraction. Low = stable sex ratios across clusters."""
    ratios = (meta_sub.groupby("cluster_label")["donor_sex_label"]
              .apply(lambda x: (x == "F").mean()))
    return float(ratios.std())


records = []
for level, col in [("subclass", "subclass_label"),
                   ("neighborhood", "neighborhood_label"),
                   ("class", "class_label")]:
    for group in meta[col].unique():
        mask = (meta[col] == group).values
        n_cells = int(mask.sum())
        meta_sub = meta[mask].reset_index(drop=True)
        n_clusters = meta_sub["cluster_label"].nunique()
        if not (1 <= n_clusters <= 20) or n_cells > 21_000:
            continue
        pca_sub = pkl_file["pca50"][mask]
        sex_sub = meta_sub["donor_sex_label"].values
        records.append({
            "level": level,
            "group": group,
            "n_cells": n_cells,
            "n_clusters": n_clusters,
            "sex_knn_score": sex_knn_score(pca_sub, sex_sub),
            "sex_ratio_std": sex_ratio_std(meta_sub),
        })

df_scan = (pd.DataFrame(records)
           .sort_values("sex_knn_score", ascending=False)
           .reset_index(drop=True))
print(df_scan.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

markers = {"subclass": "o", "neighborhood": "s", "class": "^"}
for level, grp in df_scan.groupby("level"):
    sc = ax.scatter(grp["sex_ratio_std"], grp["sex_knn_score"],
                    s=grp["n_cells"] / 100, alpha=0.7,
                    marker=markers[level], label=level)
    for _, row in grp.iterrows():
        ax.annotate(row["group"],
                    (row["sex_ratio_std"], row["sex_knn_score"]),
                    fontsize=6, alpha=0.85)

ax.set_xlabel("sex ratio std  (lower → more stable across clusters)")
ax.set_ylabel("sex kNN score  (higher → stronger batch effect)")
ax.legend(title="grouping level")
ax.set_title("Candidate subsets: batch effect vs. sex ratio stability\n(bubble size ∝ n_cells)")
fig.tight_layout()

In [ ]:
# --> seems like L2/3 ENT subclasses would work well. Select those below:

In [ ]:
mask_L23_ent = meta["subclass_label"].isin(["L2 IT ENTl", "L2 IT ENTm", "L3 IT ENT", "L2/3 IT ENTl"]).values
embd_L23_ent = embd[mask_L23_ent]
meta_L23_ent = meta[mask_L23_ent].reset_index(drop=True)

print(f"{mask_L23_ent.sum()} cells selected")

# unique color per cluster (reused by the tSNE scatter cell below)
unique_clusters = sorted(meta_L23_ent["cluster_label"].unique())
cmap = plt.get_cmap("tab20", len(unique_clusters))
cluster_color_map = {cl: cmap(i) for i, cl in enumerate(unique_clusters)}

# ── figure layout: 2 scatter + 2 bar plots ───────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.35)

ax_ct  = fig.add_subplot(gs[0, 0])
ax_sex = fig.add_subplot(gs[0, 1])
ax_bar_ct  = fig.add_subplot(gs[1, 0])
ax_bar_sex = fig.add_subplot(gs[1, 1])

# ── scatter panels ────────────────────────────────────────────────────────────
cell_colors_L23_ent = meta_L23_ent["cluster_label"].map(cluster_color_map).tolist()

ax_ct.scatter(*embd_L23_ent.T, s=1, c=cell_colors_L23_ent, alpha=0.4)
ax_ct.set_title("cell type")
ax_ct.axis("off")
ax_ct.set_aspect("equal")

ax_sex.scatter(*embd_L23_ent.T, s=1, c=meta_L23_ent["donor_sex_color"], alpha=0.4)
ax_sex.set_title("donor sex")
ax_sex.axis("off")
ax_sex.set_aspect("equal")


# ── bar: total counts per cell type ──────────────────────────────────────────
ct_counts = meta_L23_ent["cluster_label"].value_counts().sort_index()
bar_colors = [cluster_color_map[cl] for cl in ct_counts.index]

ax_bar_ct.bar(range(len(ct_counts)), ct_counts.values, color=bar_colors)
ax_bar_ct.set_xticks(range(len(ct_counts)))
ax_bar_ct.set_xticklabels(ct_counts.index, rotation=90, fontsize=6)
ax_bar_ct.set_ylabel("# cells")
ax_bar_ct.set_title("Cell count per cluster")

# ── bar: sex breakdown per cell type (stacked) ────────────────────────────────
sex_ct = (meta_L23_ent.groupby(["cluster_label", "donor_sex_label"])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(ct_counts.index))

sex_colors = (meta_L23_ent.drop_duplicates("donor_sex_label")
                      .set_index("donor_sex_label")["donor_sex_color"]
                      .to_dict())

bottom = np.zeros(len(sex_ct))
for sex in sex_ct.columns:
    ax_bar_sex.bar(range(len(sex_ct)), sex_ct[sex].values,
                   bottom=bottom, label=sex, color=sex_colors.get(sex, None))
    bottom += sex_ct[sex].values

ax_bar_sex.set_xticks(range(len(sex_ct)))
ax_bar_sex.set_xticklabels(sex_ct.index, rotation=90, fontsize=6)
ax_bar_sex.set_ylabel("# cells")
ax_bar_sex.set_title("Sex breakdown per cluster (stacked)")
ax_bar_sex.legend(title="sex", fontsize=8)

fig.suptitle(f"L2/L3 IT ENT  (n={mask_L23_ent.sum():,})", fontsize=13)
plt.show()

In [ ]:
mask_ent = meta["subclass_label"].isin(["L2 IT ENTl", "L2 IT ENTm", "L3 IT ENT", "L2/3 IT ENTl"]).values
meta_ent = meta[mask_ent].reset_index(drop=True)
cell_colors = meta_ent["cluster_label"].map(cluster_color_map).tolist()

pca_ent  = pkl_file["pca50"][mask_ent]

print(f"{mask_ent.sum()} cells — running tSNE on PCA50...")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5,5))

ax[0].scatter(*pca_ent[:, :2].T, c=cell_colors, s=0.5)
ax[1].scatter(*pca_ent[:, :2].T, c=meta_ent["donor_sex_color"], s=0.5)

# PCA has very good batch integrationl, but poor cluster separation

In [ ]:
embedder_ent = TSNE(verbose=True)
embd_ent = embedder_ent.fit(pca_ent)

In [ ]:
import matplotlib.patheffects as pe

cell_colors = meta_ent["cluster_label"].map(cluster_color_map).tolist()

# unique color per subclass
unique_subclasses = sorted(meta_ent["subclass_label"].unique())
cmap_sub = plt.get_cmap("tab10", len(unique_subclasses))
subclass_color_map = {sc: cmap_sub(i) for i, sc in enumerate(unique_subclasses)}
subclass_colors = meta_ent["subclass_label"].map(subclass_color_map).tolist()

stroke = [pe.withStroke(linewidth=2.5, foreground="white")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(*embd_ent.T, s=1, c=cell_colors, alpha=0.4)
axes[0].set_title("L2/L3 IT ENT – cell type")
axes[0].axis("off")
axes[0].set_aspect("equal")

axes[1].scatter(*embd_ent.T, s=1, c=subclass_colors, alpha=0.4)
axes[1].set_title("L2/L3 IT ENT – subclass")
axes[1].axis("off")
axes[1].set_aspect("equal")

axes[2].scatter(*embd_ent.T, s=1, c=meta_ent["donor_sex_color"], alpha=0.4)
axes[2].set_title("L2/L3 IT ENT – donor sex")
axes[2].axis("off")
axes[2].set_aspect("equal")

# cluster name labels at mean positions
ct_df = pd.DataFrame({"x": embd_ent[:, 0], "y": embd_ent[:, 1], "label": meta_ent["cluster_label"]})
for lbl, row in ct_df.groupby("label")[["x", "y"]].mean().iterrows():
    axes[0].text(row["x"], row["y"], lbl, fontsize=6, ha="center", va="center",
                 fontweight="bold", color=cluster_color_map[lbl], path_effects=stroke)

# subclass name labels at mean positions
sc_df = pd.DataFrame({"x": embd_ent[:, 0], "y": embd_ent[:, 1], "label": meta_ent["subclass_label"]})
for lbl, row in sc_df.groupby("label")[["x", "y"]].mean().iterrows():
    axes[1].text(row["x"], row["y"], lbl, fontsize=7, ha="center", va="center",
                 fontweight="bold", color=subclass_color_map[lbl], path_effects=stroke)

fig.suptitle(f"L2/L3 IT ENT  (n={mask_ent.sum():,})", fontsize=13)
fig.tight_layout()

# decent cluster purity in t-SNE, but no batch integration. Jointly the selected subclasses make sense (ENT cells going from L2l / m via L2/L3 to L3

In [ ]:
import matplotlib.colors as mcolors

save_path = os.path.join(root_path, "yao_10x_ent_l2l3")
os.makedirs(save_path, exist_ok=True)

# PCA50
np.save(os.path.join(save_path, "ent_l2l3_pca50.npy"), pca_ent)

# metadata: all existing columns + new tab20/tab10 color columns
meta_save = meta_ent.copy()
meta_save["cluster_color_new"] = meta_ent["cluster_label"].map(
    {cl: mcolors.to_hex(c) for cl, c in cluster_color_map.items()}
)
meta_save["subclass_color_new"] = meta_ent["subclass_label"].map(
    {sc: mcolors.to_hex(c) for sc, c in subclass_color_map.items()}
)
meta_save.to_csv(os.path.join(save_path, "ent_l2l3_meta.csv"), index=False)

print(f"Saved pca50:    ent_l2l3_pca50.npy   {pca_ent.shape}")
print(f"Saved metadata: ent_l2l3_meta.csv     {meta_save.shape}")

### ENT Dataset Analysis

This section mirrors the `analysis_Boris_sc` workflow on the ENT subset.
OT-based steps use a balanced, stratified female/male sample so the full sweep remains runnable.


In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from ne_aligned_utils.analysis import (
    EmbeddingViews,
    NCovAnalysisData,
    build_ot_tsne_four_block_matrix,
    compute_eot_embedding_from_arrays,
    compute_ot_plan,
    compute_ot_tsne_embedding_from_arrays,
    compute_tsne_embedding,
    normalize_ot_tsne_graph,
    plot_embedding_panels,
    plot_joint_embedding_pair,
    plot_matrix,
    plot_ot_tsne_sweep_grid,
    run_tsne_from_graph,
    sparsify_four_block_top_percentage,
)


In [ ]:
ENT_ANALYSIS_SUBCLASSES = ("L2 IT ENTl", "L2 IT ENTm", "L3 IT ENT", "L2/3 IT ENTl")
ENT_ANALYSIS_DATA_DIR = "data/yao_10x_ent_l2l3"
ENT_ANALYSIS_BASE_DIR = "data/yao_10x"
ENT_ANALYSIS_BATCH_COL = "donor_sex_label"
ENT_ANALYSIS_LABEL_COL = "cluster_label"
ENT_ANALYSIS_MAX_CELLS_PER_BATCH = 5000
ENT_ANALYSIS_RANDOM_STATE = 0
ENT_ANALYSIS_DATASET_KEY = "ent_l2l3"
ENT_ANALYSIS_DISPLAY_NAME = "ENT L2/L3"


def resolve_ent_dataset_path(data_path: str | Path = ENT_ANALYSIS_DATA_DIR) -> Path:
    path = Path(data_path).expanduser()
    candidates = [path]

    if not path.is_absolute():
        cwd = Path.cwd()
        for base in (cwd, cwd.parent, cwd.parent.parent):
            candidates.append(base / path)

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    if path.is_absolute():
        return path
    return (Path.cwd() / path).resolve()


def _allocate_group_sample_sizes(counts: pd.Series, n_target: int) -> pd.Series:
    counts = counts.astype(int).sort_index()
    n_target = min(int(n_target), int(counts.sum()))
    if n_target <= 0:
        return pd.Series(0, index=counts.index, dtype=int)

    ideal = counts / counts.sum() * n_target
    sizes = np.floor(ideal).astype(int)
    min_per_group = 1 if n_target >= len(counts) else 0
    if min_per_group:
        sizes = np.maximum(sizes, min_per_group)
    sizes = np.minimum(sizes, counts)
    sizes = pd.Series(sizes, index=counts.index, dtype=int)

    remaining = int(n_target - sizes.sum())
    if remaining > 0:
        order = (ideal - sizes).sort_values(ascending=False).index.tolist()
        while remaining > 0:
            changed = False
            for group in order:
                if sizes[group] < counts[group]:
                    sizes[group] += 1
                    remaining -= 1
                    changed = True
                    if remaining == 0:
                        break
            if not changed:
                break
    elif remaining < 0:
        order = (ideal - sizes).sort_values(ascending=True).index.tolist()
        while remaining < 0:
            changed = False
            for group in order:
                if sizes[group] > min_per_group:
                    sizes[group] -= 1
                    remaining += 1
                    changed = True
                    if remaining == 0:
                        break
            if not changed:
                break

    return sizes.astype(int)


def stratified_subsample_frame(
    frame: pd.DataFrame,
    group_col: str,
    n_target: int | None,
    random_state: int = 0,
) -> pd.DataFrame:
    if n_target is None or len(frame) <= int(n_target):
        return frame.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    counts = frame[group_col].value_counts().sort_index()
    allocations = _allocate_group_sample_sizes(counts, int(n_target))
    sampled_parts = []
    for offset, (group, n_group) in enumerate(allocations.items()):
        if n_group <= 0:
            continue
        group_frame = frame[frame[group_col] == group]
        if n_group >= len(group_frame):
            sampled = group_frame.copy()
        else:
            sampled = group_frame.sample(n=int(n_group), random_state=random_state + offset)
        sampled_parts.append(sampled)

    if not sampled_parts:
        raise ValueError("Stratified sampling produced no rows.")

    sampled_frame = pd.concat(sampled_parts, ignore_index=False)
    sampled_frame = sampled_frame.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return sampled_frame


def load_ent_dataset_arrays(
    subset_path: str | Path = ENT_ANALYSIS_DATA_DIR,
    base_path: str | Path = ENT_ANALYSIS_BASE_DIR,
    subclasses: tuple[str, ...] = ENT_ANALYSIS_SUBCLASSES,
) -> tuple[Path, np.ndarray, pd.DataFrame]:
    subset_dir = resolve_ent_dataset_path(subset_path)
    subset_pca_path = subset_dir / "ent_l2l3_pca50.npy"
    subset_meta_path = subset_dir / "ent_l2l3_meta.csv"

    if subset_pca_path.exists() and subset_meta_path.exists():
        pca50 = np.load(subset_pca_path)
        meta = pd.read_csv(subset_meta_path)
    elif "pca_ent" in globals() and "meta_ent" in globals():
        pca50 = np.asarray(globals()["pca_ent"])
        meta = globals()["meta_ent"].copy()
    else:
        base_dir = resolve_ent_dataset_path(base_path)
        base_meta_path = base_dir / "10x_metadata.csv"
        base_pca_path = base_dir / "yao_10x_v2_pca50.pkl"
        if not base_meta_path.exists() or not base_pca_path.exists():
            raise FileNotFoundError(
                "Could not find saved ENT subset files or the base yao_10x PCA/metadata files."
            )

        base_meta = pd.read_csv(base_meta_path)
        with open(base_pca_path, "rb") as file:
            compact = pickle.load(file)
        mask = base_meta["subclass_label"].isin(subclasses).to_numpy()
        pca50 = np.asarray(compact["pca50"])[mask]
        meta = base_meta.loc[mask].reset_index(drop=True)

    meta = meta.reset_index(drop=True)
    pca50 = np.asarray(pca50, dtype=float)
    if len(meta) != pca50.shape[0]:
        raise ValueError(
            f"ENT metadata rows ({len(meta)}) do not match PCA rows ({pca50.shape[0]})."
        )
    return subset_dir, pca50, meta


def load_ent_analysis_data(
    data_path: str | Path = ENT_ANALYSIS_DATA_DIR,
    base_path: str | Path = ENT_ANALYSIS_BASE_DIR,
    batch_col: str = ENT_ANALYSIS_BATCH_COL,
    label_col: str = ENT_ANALYSIS_LABEL_COL,
    max_cells_per_batch: int | None = ENT_ANALYSIS_MAX_CELLS_PER_BATCH,
    random_state: int = ENT_ANALYSIS_RANDOM_STATE,
) -> tuple[NCovAnalysisData, pd.DataFrame, Path]:
    resolved_path, pca50, meta = load_ent_dataset_arrays(
        subset_path=data_path,
        base_path=base_path,
        subclasses=ENT_ANALYSIS_SUBCLASSES,
    )
    meta = meta.copy()
    meta["_row_index"] = np.arange(len(meta), dtype=int)

    batch_values = meta[batch_col].astype(str)
    unique_batches = sorted(batch_values.dropna().unique())
    preferred_batches = [value for value in ("F", "M") if value in unique_batches]
    if len(preferred_batches) == 2:
        batch_order = preferred_batches
    else:
        if len(unique_batches) < 2:
            raise ValueError(f"Expected at least two batches in {batch_col}, found {unique_batches}.")
        batch_order = unique_batches[:2]

    sampled_batches = []
    for batch_idx, batch_name in enumerate(batch_order):
        batch_frame = meta.loc[batch_values == batch_name].copy()
        sampled_batch = stratified_subsample_frame(
            batch_frame,
            group_col=label_col,
            n_target=max_cells_per_batch,
            random_state=random_state + 1000 * batch_idx,
        )
        sampled_batch[batch_col] = batch_name
        sampled_batches.append(sampled_batch)

    meta_first, meta_second = sampled_batches
    joint_meta = pd.concat([meta_first, meta_second], ignore_index=True)

    label_values = joint_meta[label_col].astype(str)
    cell_type_names = tuple(sorted(label_values.unique()))
    label_to_index = {name: idx for idx, name in enumerate(cell_type_names)}

    x1 = pca50[meta_first["_row_index"].to_numpy(dtype=int)]
    x2 = pca50[meta_second["_row_index"].to_numpy(dtype=int)]
    labels1 = meta_first[label_col].astype(str).map(label_to_index).to_numpy(dtype=int)
    labels2 = meta_second[label_col].astype(str).map(label_to_index).to_numpy(dtype=int)

    data = NCovAnalysisData(
        x1=np.asarray(x1, dtype=float),
        x2=np.asarray(x2, dtype=float),
        labels1=np.asarray(labels1, dtype=int),
        labels2=np.asarray(labels2, dtype=int),
        cell_type_names=cell_type_names,
        dataset_names=(f"{ENT_ANALYSIS_DATASET_KEY}-{batch_order[0]}", f"{ENT_ANALYSIS_DATASET_KEY}-{batch_order[1]}"),
    )
    return data, joint_meta.reset_index(drop=True), resolved_path


ent_sc, ent_sampled_meta, ent_analysis_path = load_ent_analysis_data()
print(f"Using {ENT_ANALYSIS_DISPLAY_NAME} analysis data from {ent_analysis_path}")
print(
    f"batch sizes: {ent_sc.short_dataset_names[0]}={ent_sc.x1.shape[0]:,}, "
    f"{ent_sc.short_dataset_names[1]}={ent_sc.x2.shape[0]:,}; "
    f"clusters={len(ent_sc.cell_type_names)}; total={ent_sc.joint_data.shape[0]:,}"
)
ent_sc.dataset_names, ent_sc.short_dataset_names


In [ ]:
ent_pca_embeddings = EmbeddingViews(
    name="PCA50",
    first=ent_sc.x1,
    second=ent_sc.x2,
    joint=ent_sc.joint_data,
)
fig, _ = plot_embedding_panels(ent_pca_embeddings, ent_sc, color_by="batch")


In [ ]:
fig, _ = plot_embedding_panels(ent_pca_embeddings, ent_sc, color_by="cell_type")


In [ ]:
ent_tsne_embeddings = compute_tsne_embedding(ent_pca_embeddings, random_state=0)
fig, _ = plot_embedding_panels(ent_tsne_embeddings, ent_sc, color_by="batch")


In [ ]:
fig, _ = plot_embedding_panels(ent_tsne_embeddings, ent_sc, color_by="cell_type")


In [ ]:
ent_paper_eot_eps_times = 1.0  # Paper-style epsilon: median cross-batch squared distance.

ent_eot_embeddings = compute_eot_embedding_from_arrays(
    ent_sc.x1,
    ent_sc.x2,
    t=0,
    eps_times=ent_paper_eot_eps_times,
)
fig, _ = plot_embedding_panels(
    ent_eot_embeddings,
    ent_sc,
    color_by="batch",
    point_size=1.5,
    alpha=0.45,
    standardize=True,
    aspect="auto",
)


In [ ]:
fig, _ = plot_embedding_panels(
    ent_eot_embeddings,
    ent_sc,
    color_by="cell_type",
    point_size=1.5,
    alpha=0.45,
    standardize=True,
    aspect="auto",
)


### OT-TSNE


#### Step 1: Calculate OT Plan


In [ ]:
ent_ot_plan = compute_ot_plan(ent_sc.x1, ent_sc.x2, eps_times=1.0, tol=1e-12, max_iter=int(1e4))
print(f"shape={ent_ot_plan.shape}, sum={ent_ot_plan.sum():.6f}, nnz={np.count_nonzero(ent_ot_plan)}")
fig, _ = plot_matrix(ent_ot_plan, title="ENT OT plan")


#### Step 2: Build 4 Block Matrix


In [ ]:
ent_ot_tsne_four_block = build_ot_tsne_four_block_matrix(ent_ot_plan)
print(
    f"shape={ent_ot_tsne_four_block.shape}, sum={ent_ot_tsne_four_block.sum():.6f}, "
    f"nnz={np.count_nonzero(ent_ot_tsne_four_block)}"
)
fig, _ = plot_matrix(ent_ot_tsne_four_block, title="ENT OT-tSNE 4-block matrix")


#### Step 3: Sparsify 4 Block Matrix (top 0.5% neighbors)


In [ ]:
ent_ot_tsne_sparse_four_block = sparsify_four_block_top_percentage(
    ent_ot_tsne_four_block,
    n0=ent_sc.x1.shape[0],
    n1=ent_sc.x2.shape[0],
    percentage=0.005,
    ensure_nonempty=True,
    binarize=True,
)
print(
    f"shape={ent_ot_tsne_sparse_four_block.shape}, sum={ent_ot_tsne_sparse_four_block.sum():.6f}, "
    f"nnz={np.count_nonzero(ent_ot_tsne_sparse_four_block)}"
)
fig, _ = plot_matrix(
    ent_ot_tsne_sparse_four_block,
    title="Sparse ENT OT-tSNE 4-block matrix (binarized)",
)


#### Step 4: Normalize


In [ ]:
ent_ot_tsne_graph = normalize_ot_tsne_graph(ent_ot_tsne_sparse_four_block)
print(
    f"shape={ent_ot_tsne_graph.shape}, sum={ent_ot_tsne_graph.sum():.6f}, "
    f"nnz={np.count_nonzero(ent_ot_tsne_graph)}"
)
fig, _ = plot_matrix(ent_ot_tsne_graph, title="Normalized ENT OT-tSNE graph")


#### Step 5: t-SNE


In [ ]:
ent_ot_tsne_embedding = run_tsne_from_graph(ent_ot_tsne_graph, random_state=0, n_epochs=200)
ent_ot_tsne_embedding.shape


In [ ]:
fig, _ = plot_joint_embedding_pair(
    ent_ot_tsne_embedding,
    ent_sc,
    point_size=2.0,
    alpha=0.7,
    aspect="auto",
)


#### OT-tSNE Sweep


In [ ]:
ent_ot_tsne_sweep_embeddings, fig, _ = plot_ot_tsne_sweep_grid(
    ent_sc,
    percentages=(0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2),
    eps_values=(1.0, 10.0, 100.0, 300.0),
    point_size=1.5,
    alpha=0.7,
    binarize=True,
    aspect="auto",
)


In [ ]:
ent_ot_tsne_sweep_embeddings, fig, _ = plot_ot_tsne_sweep_grid(
    ent_sc,
    percentages=(0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.4),
    eps_values=(0.001, 0.1, 1.0, 10.0, 100.0, 300.0),
    point_size=1.5,
    alpha=0.7,
    binarize=False,
    aspect="auto",
)


#### Benchmarking


### Embedding Metrics


In [ ]:
from pathlib import Path
import subprocess
import sys


def _embedding_metric_format_token(value: float | int | str) -> str:
    text = str(value).replace("-", "m").replace(".", "p")
    return "".join(char if char.isalnum() or char == "_" else "_" for char in text)


embedding_metric_percentages = (0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2)
embedding_metric_eps_values = (1.0, 10.0, 100.0, 300.0)
embedding_metric_k = 50
embedding_metric_random_state = 0
embedding_metric_n_epochs = 200
embedding_metric_binarize = True
embedding_metric_eot_t = 0
embedding_metric_eot_source = "raw_arrays"

embedding_metric_cache_dir = resolve_ent_dataset_path()
embedding_metric_variant = "binarized" if embedding_metric_binarize else "weighted"
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)

config_token = "__".join(
    [
        f"variant_{_embedding_metric_format_token(embedding_metric_variant)}",
        "n_per_batch_5000",
        "subsample_seed_0",
        f"embed_seed_{_embedding_metric_format_token(embedding_metric_random_state)}",
        f"epochs_{_embedding_metric_format_token(embedding_metric_n_epochs)}",
        f"eot_t_{_embedding_metric_format_token(embedding_metric_eot_t)}",
        f"eot_source_{_embedding_metric_format_token(embedding_metric_eot_source)}",
    ]
)
embedding_metric_task_root = embedding_metric_cache_dir / "joint_ot_tsne_task_cache" / config_token

expected_task_paths = []
for eps_times in embedding_metric_eps_values:
    eps_token = _embedding_metric_format_token(f"{eps_times:g}")
    expected_task_paths.append(embedding_metric_task_root / f"eot_t_{embedding_metric_eot_t}_eps_times_{eps_token}.npz")
    for percentage in embedding_metric_percentages:
        pct_token = _embedding_metric_format_token(f"{percentage:g}")
        expected_task_paths.append(
            embedding_metric_task_root / f"ot_tsne_eps_times_{eps_token}_pct_{pct_token}.npz"
        )

missing_task_paths = [path for path in expected_task_paths if not path.exists()]
if embedding_metric_cache_path.exists():
    print(f"Final sweep cache already available at {embedding_metric_cache_path}")
elif not missing_task_paths:
    aggregate_script = Path.cwd() / "scripts" / "yao_10x_joint_ot_tsne_sweep.py"
    if not aggregate_script.exists():
        raise FileNotFoundError(
            f"Could not find aggregation script at {aggregate_script}."
        )

    command = [
        sys.executable,
        str(aggregate_script),
        "aggregate",
        "--data-path",
        str(ENT_ANALYSIS_DATA_DIR),
        "--base-path",
        str(ENT_ANALYSIS_BASE_DIR),
        "--subsample-random-state",
        "0",
        "--embedding-random-state",
        str(embedding_metric_random_state),
        "--n-epochs",
        str(embedding_metric_n_epochs),
        "--eot-t",
        str(embedding_metric_eot_t),
        "--eot-source",
        embedding_metric_eot_source,
        "--metric-k",
        str(embedding_metric_k),
        "--max-cells-per-batch",
        "5000",
        "--binarize",
        "--output-cache",
        str(embedding_metric_cache_path),
        "--eps-values",
        *(f"{value:g}" for value in embedding_metric_eps_values),
        "--percentages",
        *(f"{value:g}" for value in embedding_metric_percentages),
    ]
    print("All Slurm sweep task files are present; aggregating final cache...")
    subprocess.run(command, check=True)
elif embedding_metric_task_root.exists():
    print(
        f"Slurm sweep task directory exists but {len(missing_task_paths)} task files are still missing; "
        "leaving the next cell to compute or wait for the remaining jobs."
    )
else:
    print(
        "No aggregated cache or Slurm task directory found yet; "
        "the next cell will fall back to computing the sweep locally."
    )


In [ ]:
from pathlib import Path

import numpy as np

embedding_metric_percentages = (0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2)
embedding_metric_eps_values = (1.0, 10.0, 100.0, 300.0)
embedding_metric_k = 50
embedding_metric_random_state = 0
embedding_metric_n_epochs = 200
embedding_metric_binarize = True
embedding_metric_eot_t = 0
embedding_metric_eot_source = "raw_arrays"

embedding_metric_cache_dir = resolve_ent_dataset_path()
embedding_metric_cache_dir.mkdir(parents=True, exist_ok=True)
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = "binarized" if embedding_metric_binarize else "weighted"
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)

expected_shape = (
    len(embedding_metric_eps_values),
    len(embedding_metric_percentages),
    ent_sc.joint_data.shape[0],
    2,
)
expected_eot_shape = (len(embedding_metric_eps_values), ent_sc.joint_data.shape[0], 2)

if embedding_metric_cache_path.exists():
    with np.load(embedding_metric_cache_path) as cache:
        cached_eps_values = tuple(float(value) for value in cache["eps_values"])
        cached_percentages = tuple(float(value) for value in cache["percentages"])
        cached_binarize = bool(cache["binarize"].item())
        cached_eot_t = int(cache["eot_t"])
        cached_eot_source = str(cache["eot_source"].item()) if "eot_source" in cache else ""
        ent_ot_tsne_metric_embeddings = np.asarray(cache["ot_tsne_embeddings"], dtype=float)
        ent_eot_metric_embeddings = np.asarray(cache["eot_embeddings"], dtype=float)

    cache_matches_request = (
        cached_eps_values == embedding_metric_eps_values
        and cached_percentages == embedding_metric_percentages
        and cached_binarize == embedding_metric_binarize
        and cached_eot_t == embedding_metric_eot_t
        and cached_eot_source == embedding_metric_eot_source
        and ent_ot_tsne_metric_embeddings.shape == expected_shape
        and ent_eot_metric_embeddings.shape == expected_eot_shape
    )
else:
    cache_matches_request = False

if not cache_matches_request:
    print("Cache missing or incompatible; computing joint OT-tSNE metric embeddings...")
    ent_ot_tsne_metric_embeddings = np.empty(expected_shape, dtype=float)
    ent_eot_metric_embeddings = np.empty(expected_eot_shape, dtype=float)
    total_progress_steps = len(embedding_metric_eps_values) * (1 + len(embedding_metric_percentages))
    progress_step = 0

    for eps_idx, eps_times in enumerate(embedding_metric_eps_values):
        progress_step += 1
        print(
            f"[{progress_step}/{total_progress_steps}] Computing paper-style EOT t={embedding_metric_eot_t} "
            f"from raw arrays for eps_times={eps_times:g}"
        )
        ent_eot_metric_embeddings[eps_idx] = compute_eot_embedding_from_arrays(
            ent_sc.x1,
            ent_sc.x2,
            t=embedding_metric_eot_t,
            eps_times=eps_times,
        )

        for pct_idx, percentage in enumerate(embedding_metric_percentages):
            progress_step += 1
            print(
                f"[{progress_step}/{total_progress_steps}] Computing joint OT-tSNE embedding "
                f"for eps_times={eps_times:g}, top {100.0 * percentage:g}% neighbors"
            )
            ent_ot_tsne_metric_embeddings[eps_idx, pct_idx] = compute_ot_tsne_embedding_from_arrays(
                ent_sc.x1,
                ent_sc.x2,
                eps_times=eps_times,
                percentage=percentage,
                binarize=embedding_metric_binarize,
                random_state=embedding_metric_random_state,
                n_epochs=embedding_metric_n_epochs,
            )

    np.savez_compressed(
        embedding_metric_cache_path,
        eps_values=np.asarray(embedding_metric_eps_values, dtype=float),
        percentages=np.asarray(embedding_metric_percentages, dtype=float),
        ot_tsne_embeddings=ent_ot_tsne_metric_embeddings,
        eot_embeddings=ent_eot_metric_embeddings,
        binarize=np.asarray(embedding_metric_binarize),
        metric_k=np.asarray(embedding_metric_k, dtype=int),
        eot_t=np.asarray(embedding_metric_eot_t, dtype=int),
        eot_source=np.asarray(embedding_metric_eot_source),
        random_state=np.asarray(embedding_metric_random_state, dtype=int),
        n_epochs=np.asarray(embedding_metric_n_epochs, dtype=int),
    )
    print(f"Saved joint OT-tSNE metric embeddings to {embedding_metric_cache_path}")
else:
    print(f"Loaded joint OT-tSNE metric embeddings from {embedding_metric_cache_path}")

print(
    f"variant={embedding_metric_variant}, eps_times={embedding_metric_eps_values}, "
    f"percentages={embedding_metric_percentages}, k={embedding_metric_k}, "
    f"EOT source={embedding_metric_eot_source}, "
    f"OT grid shape={ent_ot_tsne_metric_embeddings.shape}, "
    f"EOT grid shape={ent_eot_metric_embeddings.shape}"
)


In [ ]:
import numpy as np

embedding_metric_cache_dir = resolve_ent_dataset_path()
embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the previous embedding cache cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)

embedding_metric_selected_eot_t = 0  # Change this before running the cell.
embedding_metric_selected_eot_source = "raw_arrays"

embedding_metric_selected_eot_embeddings = np.empty(
    (len(embedding_metric_eps_values), ent_sc.joint_data.shape[0], 2),
    dtype=float,
)
for eps_idx, eps_times in enumerate(embedding_metric_eps_values):
    print(
        f"[{eps_idx + 1}/{len(embedding_metric_eps_values)}] Computing paper-style EOT t="
        f"{embedding_metric_selected_eot_t} from raw arrays for eps_times={eps_times:g}"
    )
    embedding_metric_selected_eot_embeddings[eps_idx] = compute_eot_embedding_from_arrays(
        ent_sc.x1,
        ent_sc.x2,
        t=embedding_metric_selected_eot_t,
        eps_times=eps_times,
    )

embedding_metric_selected_variant = embedding_metric_variant
embedding_metric_selected_eps_values = embedding_metric_eps_values.copy()

print(
    f"Prepared paper-style EOT reference embeddings for t={embedding_metric_selected_eot_t} "
    f"with shape {embedding_metric_selected_eot_embeddings.shape}"
)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ne_aligned_utils.benchmarking import (
    compute_shared_dataset_scale_bounds,
    plot_metric_heatmap_grid,
)
from ne_aligned_utils.metrics import (
    compute_knn_other_batch_fraction,
    compute_knn_same_cell_type_fraction,
)

embedding_metric_cache_dir = resolve_ent_dataset_path()
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the previous cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)
    embedding_metric_percentages = np.asarray(cache["percentages"], dtype=float)
    ent_ot_tsne_metric_embeddings = np.asarray(cache["ot_tsne_embeddings"], dtype=float)
    embedding_metric_binarize = bool(cache["binarize"].item())
    embedding_metric_k = int(cache["metric_k"])
    embedding_metric_random_state = int(cache["random_state"]) if "random_state" in cache else 0

expected_selected_eot_source = "raw_arrays"
ent_eot_metric_embeddings = globals().get("embedding_metric_selected_eot_embeddings")
if ent_eot_metric_embeddings is None:
    raise RuntimeError(
        "No selected EOT embeddings found in memory. "
        "Run the previous EOT selection cell first."
    )

embedding_metric_eot_t = int(globals().get("embedding_metric_selected_eot_t", 0))
embedding_metric_selected_source = globals().get("embedding_metric_selected_eot_source")
if embedding_metric_selected_source != expected_selected_eot_source:
    raise RuntimeError(
        "Selected EOT embeddings were not computed from raw arrays. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_variant = globals().get("embedding_metric_selected_variant")
if embedding_metric_selected_variant != embedding_metric_variant:
    raise RuntimeError(
        "Selected EOT embeddings were computed for a different variant. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_eps_values = globals().get("embedding_metric_selected_eps_values")
if embedding_metric_selected_eps_values is None or not np.array_equal(
    np.asarray(embedding_metric_selected_eps_values, dtype=float),
    embedding_metric_eps_values,
):
    raise RuntimeError(
        "Selected EOT embeddings do not match the cached eps grid. "
        "Re-run the previous EOT selection cell."
    )

ent_eot_metric_embeddings = np.asarray(ent_eot_metric_embeddings, dtype=float)
expected_eot_shape = (
    len(embedding_metric_eps_values),
    ent_sc.joint_data.shape[0],
    2,
)
if ent_eot_metric_embeddings.shape != expected_eot_shape:
    raise ValueError(
        f"Expected selected EOT embeddings with shape {expected_eot_shape}, "
        f"got {ent_eot_metric_embeddings.shape}."
    )

metric_base_embeddings = globals().get("ent_pca_embeddings")
if metric_base_embeddings is None:
    metric_base_embeddings = EmbeddingViews(
        name="PCA50",
        first=ent_sc.x1,
        second=ent_sc.x2,
        joint=ent_sc.joint_data,
    )

metric_joint_tsne_embeddings = globals().get("ent_tsne_embeddings")
if metric_joint_tsne_embeddings is None:
    metric_joint_tsne_embeddings = compute_tsne_embedding(
        metric_base_embeddings,
        random_state=embedding_metric_random_state,
    )

metric_specs = {
    "same_cell_type": {
        "label": "kNN Same-Cell-Type Fraction",
        "fn": lambda emb: compute_knn_same_cell_type_fraction(
            emb,
            ent_sc.joint_labels,
            k=embedding_metric_k,
        ),
        "cmap": "viridis",
    },
    "other_batch": {
        "label": "Shared-Type Batch Mixing Score",
        "fn": lambda emb: compute_knn_other_batch_fraction(
            emb,
            ent_sc.batch_ids,
            ent_sc.joint_labels,
            k=embedding_metric_k,
        ),
        "cmap": "viridis",
    },
}

eot_reference_label = f"EOT {embedding_metric_eot_t:g}"

heatmap_data_by_metric = {}
for metric_name, spec in metric_specs.items():
    matrix = np.empty((len(embedding_metric_eps_values), len(embedding_metric_percentages)), dtype=float)
    eot_column = np.empty(len(embedding_metric_eps_values), dtype=float)
    joint_tsne_value = float(spec["fn"](metric_joint_tsne_embeddings))
    joint_pca_value = float(spec["fn"](metric_base_embeddings))

    for eps_idx in range(len(embedding_metric_eps_values)):
        eot_column[eps_idx] = spec["fn"](ent_eot_metric_embeddings[eps_idx])
        for pct_idx in range(len(embedding_metric_percentages)):
            matrix[eps_idx, pct_idx] = spec["fn"](
                ent_ot_tsne_metric_embeddings[eps_idx, pct_idx]
            )

    reference_columns = [
        {"label": eot_reference_label, "values": eot_column},
        {
            "label": "joint t-SNE",
            "values": np.full(len(embedding_metric_eps_values), joint_tsne_value, dtype=float),
        },
        {
            "label": "joint PCA",
            "values": np.full(len(embedding_metric_eps_values), joint_pca_value, dtype=float),
        },
    ]

    heatmap_data_by_metric[metric_name] = {
        "metric_name": metric_name,
        "metric_label": spec["label"],
        "eps_times": embedding_metric_eps_values,
        "pct_values": 100.0 * embedding_metric_percentages,
        "eot_t": embedding_metric_eot_t,
        "dataset_names": (ENT_ANALYSIS_DATASET_KEY,),
        "datasets": {
            ENT_ANALYSIS_DATASET_KEY: {
                "top_pct_matrix": matrix,
                "reference_columns": reference_columns,
                "eot_column": eot_column,
            }
        },
        "y_labels": [f"{eps:g}" for eps in embedding_metric_eps_values],
        "top_pct_column_count": int(len(embedding_metric_percentages)),
    }

embedding_metric_scale_bounds = compute_shared_dataset_scale_bounds(heatmap_data_by_metric)
embedding_metric_heatmap_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_knn_metric_heatmaps_{embedding_metric_variant}.npz"
)

for metric_name, heatmap_data in heatmap_data_by_metric.items():
    figure_path = (
        embedding_metric_figure_dir
        / f"{metric_name}_joint_ot_tsne_eps_pct_heatmap_{embedding_metric_variant}.png"
    )
    fig, axes = plot_metric_heatmap_grid(
        heatmap_data,
        title=(
            f"{heatmap_data['metric_label']} on joint {ENT_ANALYSIS_DISPLAY_NAME} OT-tSNE embeddings "
            f"({embedding_metric_variant}, k={embedding_metric_k})"
        ),
        cmap=metric_specs[metric_name]["cmap"],
        save_path=figure_path,
        scale_bounds_by_dataset=embedding_metric_scale_bounds,
        annotate_cells=True,
    )
    axes[ENT_ANALYSIS_DATASET_KEY]["main"].set_ylabel("eps_times")
    plt.show()
    print(f"Saved {metric_name} heatmap figure to {figure_path}")

np.savez_compressed(
    embedding_metric_heatmap_cache_path,
    eps_values=embedding_metric_eps_values,
    percentages=embedding_metric_percentages,
    pct_values_percent=100.0 * embedding_metric_percentages,
    metric_k=np.asarray(embedding_metric_k, dtype=int),
    eot_t=np.asarray(embedding_metric_eot_t, dtype=int),
    eot_source=np.asarray(expected_selected_eot_source),
    binarize=np.asarray(embedding_metric_binarize),
    same_cell_type_matrix=heatmap_data_by_metric["same_cell_type"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["top_pct_matrix"],
    same_cell_type_eot=heatmap_data_by_metric["same_cell_type"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["eot_column"],
    same_cell_type_joint_tsne=heatmap_data_by_metric["same_cell_type"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["reference_columns"][1]["values"],
    same_cell_type_joint_pca=heatmap_data_by_metric["same_cell_type"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["reference_columns"][2]["values"],
    other_batch_matrix=heatmap_data_by_metric["other_batch"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["top_pct_matrix"],
    other_batch_eot=heatmap_data_by_metric["other_batch"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["eot_column"],
    other_batch_joint_tsne=heatmap_data_by_metric["other_batch"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["reference_columns"][1]["values"],
    other_batch_joint_pca=heatmap_data_by_metric["other_batch"]["datasets"][ENT_ANALYSIS_DATASET_KEY]["reference_columns"][2]["values"],
)
print(f"Saved embedding-metric heatmap matrices to {embedding_metric_heatmap_cache_path}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def standardize_embedding_for_display(embedding, eps=0.0):
    embedding = np.asarray(embedding, dtype=float)
    centered = embedding - embedding.mean(axis=0, keepdims=True)
    scale = embedding.std(axis=0, keepdims=True)
    scale = np.where(scale <= eps, 1.0, scale)
    return centered / scale


def compute_display_bounds(embeddings, lower=1.0, upper=99.0, pad_fraction=0.08):
    x_values = np.asarray(embeddings[..., 0], dtype=float).ravel()
    y_values = np.asarray(embeddings[..., 1], dtype=float).ravel()
    x_min, x_max = np.percentile(x_values, [lower, upper])
    y_min, y_max = np.percentile(y_values, [lower, upper])
    x_pad = pad_fraction * max(float(x_max - x_min), 1.0)
    y_pad = pad_fraction * max(float(y_max - y_min), 1.0)
    return (float(x_min - x_pad), float(x_max + x_pad)), (float(y_min - y_pad), float(y_max + y_pad))


embedding_metric_cache_dir = resolve_ent_dataset_path()
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the embedding cache cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)
    embedding_metric_percentages = np.asarray(cache["percentages"], dtype=float)
    ent_ot_tsne_metric_embeddings = np.asarray(cache["ot_tsne_embeddings"], dtype=float)

ent_ot_tsne_display_embeddings = np.empty_like(ent_ot_tsne_metric_embeddings)
for eps_idx in range(len(embedding_metric_eps_values)):
    for pct_idx in range(len(embedding_metric_percentages)):
        ent_ot_tsne_display_embeddings[eps_idx, pct_idx] = standardize_embedding_for_display(
            ent_ot_tsne_metric_embeddings[eps_idx, pct_idx]
        )

cell_type_palette = np.array(
    [plt.get_cmap("tab20", len(ent_sc.cell_type_names))(i) for i in range(len(ent_sc.cell_type_names))]
)
batch_palette = np.array(["tab:red", "tab:blue"], dtype=object)
plot_order = np.random.default_rng(0).permutation(ent_sc.joint_data.shape[0])
point_size = 2.0
alpha = 0.7

fig_cell, axes_cell = plt.subplots(
    len(embedding_metric_eps_values),
    len(embedding_metric_percentages),
    figsize=(3.0 * len(embedding_metric_percentages), 3.0 * len(embedding_metric_eps_values)),
    constrained_layout=True,
    squeeze=False,
)

fig_batch, axes_batch = plt.subplots(
    len(embedding_metric_eps_values),
    len(embedding_metric_percentages),
    figsize=(3.0 * len(embedding_metric_percentages), 3.0 * len(embedding_metric_eps_values)),
    constrained_layout=True,
    squeeze=False,
)

for row_idx, eps_times in enumerate(embedding_metric_eps_values):
    row_x_limits, row_y_limits = compute_display_bounds(
        ent_ot_tsne_display_embeddings[row_idx],
        lower=0.5,
        upper=99.5,
        pad_fraction=0.08,
    )

    for pct_idx, percentage in enumerate(embedding_metric_percentages):
        embedding = ent_ot_tsne_display_embeddings[row_idx, pct_idx]
        ax_cell = axes_cell[row_idx, pct_idx]
        ax_batch = axes_batch[row_idx, pct_idx]

        ax_cell.scatter(
            embedding[plot_order, 0],
            embedding[plot_order, 1],
            s=point_size,
            c=cell_type_palette[ent_sc.joint_labels][plot_order],
            alpha=alpha,
        )
        ax_batch.scatter(
            embedding[plot_order, 0],
            embedding[plot_order, 1],
            s=point_size,
            c=batch_palette[ent_sc.batch_ids][plot_order],
            alpha=alpha,
        )

        if row_idx == 0:
            ax_cell.set_title(f"p={100.0 * percentage:g}%")
            ax_batch.set_title(f"p={100.0 * percentage:g}%")

        if pct_idx == 0:
            ax_cell.set_ylabel(f"eps={eps_times:g}")
            ax_batch.set_ylabel(f"eps={eps_times:g}")

        for ax in (ax_cell, ax_batch):
            ax.set_xlim(*row_x_limits)
            ax.set_ylim(*row_y_limits)
            ax.set_aspect("equal")
            ax.axis("off")

fig_cell.suptitle(
    f"Joint {ENT_ANALYSIS_DISPLAY_NAME} OT-tSNE embeddings by cell type ({embedding_metric_variant}, standardized for display)"
)
fig_batch.suptitle(
    f"Joint {ENT_ANALYSIS_DISPLAY_NAME} OT-tSNE embeddings by batch ({embedding_metric_variant}, standardized for display)"
)

ot_cell_figure_path = (
    embedding_metric_figure_dir
    / f"joint_ot_tsne_embedding_grid_cell_type_{embedding_metric_variant}.png"
)
ot_batch_figure_path = (
    embedding_metric_figure_dir
    / f"joint_ot_tsne_embedding_grid_batch_{embedding_metric_variant}.png"
)
fig_cell.savefig(ot_cell_figure_path, bbox_inches="tight")
fig_batch.savefig(ot_batch_figure_path, bbox_inches="tight")
plt.show()
print(f"Saved OT-tSNE cell-type embedding grid to {ot_cell_figure_path}")
print(f"Saved OT-tSNE batch embedding grid to {ot_batch_figure_path}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def standardize_embedding_for_display(embedding, eps=1e-12):
    embedding = np.asarray(embedding, dtype=float)
    centered = embedding - embedding.mean(axis=0, keepdims=True)
    scale = embedding.std(axis=0, keepdims=True)
    scale = np.where(scale < eps, 1.0, scale)
    return centered / scale


def compute_display_bounds(embeddings, lower=1.0, upper=99.0, pad_fraction=0.08):
    x_values = np.asarray(embeddings[..., 0], dtype=float).ravel()
    y_values = np.asarray(embeddings[..., 1], dtype=float).ravel()
    x_min, x_max = np.percentile(x_values, [lower, upper])
    y_min, y_max = np.percentile(y_values, [lower, upper])
    x_pad = pad_fraction * max(float(x_max - x_min), 1.0)
    y_pad = pad_fraction * max(float(y_max - y_min), 1.0)
    return (float(x_min - x_pad), float(x_max + x_pad)), (float(y_min - y_pad), float(y_max + y_pad))


def compute_embedding_span(embedding, lower=0.5, upper=99.5):
    embedding = np.asarray(embedding, dtype=float)
    quantiles = np.percentile(embedding, [lower, upper], axis=0)
    return float(np.max(quantiles[1] - quantiles[0]))


embedding_metric_cache_dir = resolve_ent_dataset_path()
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the embedding cache cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)

expected_selected_eot_source = "raw_arrays"
ent_eot_metric_embeddings = globals().get("embedding_metric_selected_eot_embeddings")
if ent_eot_metric_embeddings is None:
    raise RuntimeError(
        "No selected EOT embeddings found in memory. "
        "Run the previous EOT selection cell first."
    )

embedding_metric_eot_t = int(globals().get("embedding_metric_selected_eot_t", 0))
embedding_metric_selected_source = globals().get("embedding_metric_selected_eot_source")
if embedding_metric_selected_source != expected_selected_eot_source:
    raise RuntimeError(
        "Selected EOT embeddings were not computed from raw arrays. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_variant = globals().get("embedding_metric_selected_variant")
if embedding_metric_selected_variant != embedding_metric_variant:
    raise RuntimeError(
        "Selected EOT embeddings were computed for a different variant. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_eps_values = globals().get("embedding_metric_selected_eps_values")
if embedding_metric_selected_eps_values is None or not np.array_equal(
    np.asarray(embedding_metric_selected_eps_values, dtype=float),
    embedding_metric_eps_values,
):
    raise RuntimeError(
        "Selected EOT embeddings do not match the cached eps grid. "
        "Re-run the previous EOT selection cell."
    )

ent_eot_metric_embeddings = np.asarray(ent_eot_metric_embeddings, dtype=float)
expected_eot_shape = (
    len(embedding_metric_eps_values),
    ent_sc.joint_data.shape[0],
    2,
)
if ent_eot_metric_embeddings.shape != expected_eot_shape:
    raise ValueError(
        f"Expected selected EOT embeddings with shape {expected_eot_shape}, "
        f"got {ent_eot_metric_embeddings.shape}."
    )

ent_eot_display_embeddings = np.empty_like(ent_eot_metric_embeddings)
for row_idx in range(len(embedding_metric_eps_values)):
    ent_eot_display_embeddings[row_idx] = standardize_embedding_for_display(
        ent_eot_metric_embeddings[row_idx]
    )

cell_type_palette = np.array(
    [plt.get_cmap("tab20", len(ent_sc.cell_type_names))(i) for i in range(len(ent_sc.cell_type_names))]
)
batch_palette = np.array(["tab:red", "tab:blue"], dtype=object)
plot_order = np.random.default_rng(0).permutation(ent_sc.joint_data.shape[0])
point_size = 2.0
alpha = 0.7

(x_limits, y_limits) = compute_display_bounds(ent_eot_display_embeddings)
reference_spans = np.array(
    [compute_embedding_span(embedding) for embedding in ent_eot_metric_embeddings],
    dtype=float,
)

fig, axes = plt.subplots(
    len(embedding_metric_eps_values),
    2,
    figsize=(12, 3.4 * len(embedding_metric_eps_values)),
    constrained_layout=True,
    squeeze=False,
)

for row_idx, eps_times in enumerate(embedding_metric_eps_values):
    embedding = ent_eot_display_embeddings[row_idx]

    axes[row_idx, 0].scatter(
        embedding[plot_order, 0],
        embedding[plot_order, 1],
        s=point_size,
        c=cell_type_palette[ent_sc.joint_labels][plot_order],
        alpha=alpha,
    )
    axes[row_idx, 1].scatter(
        embedding[plot_order, 0],
        embedding[plot_order, 1],
        s=point_size,
        c=batch_palette[ent_sc.batch_ids][plot_order],
        alpha=alpha,
    )

    if row_idx == 0:
        axes[row_idx, 0].set_title("Cell type")
        axes[row_idx, 1].set_title("Batch")

    axes[row_idx, 0].set_ylabel(
        f"eps_times={eps_times:g}\nspan={reference_spans[row_idx]:.2f}"
    )

    for ax in axes[row_idx]:
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_aspect("equal")
        ax.axis("off")

fig.suptitle(
    f"Paper-style EOT t={embedding_metric_eot_t} reference embeddings from raw arrays used in the embedding heatmaps"
)

eot_reference_figure_path = (
    embedding_metric_figure_dir
    / f"eot_t{embedding_metric_eot_t}_raw_arrays_reference_embeddings_{embedding_metric_variant}.png"
)
fig.savefig(eot_reference_figure_path, bbox_inches="tight")
plt.show()
print(f"Saved EOT reference embedding figure to {eot_reference_figure_path}")
